### Importing Modules

In [1]:
import pandas as pd,numpy as np,json
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv('./clustered_dataset/umap_clustered_dataset_iter_02.csv')

In [3]:
data.head()

,Pattern Name,Problem,Context,Solution,Result,Uses,cluster
0,LLM-Tool Orchestration and Augmentation,Large Language Models (LLMs) are inherently li...,"AI systems, particularly LLMs, are tasked with...",The LLM acts as an intelligent orchestrator or...,Significantly expands the LLM's capabilities b...,"Open-domain question answering, fact-checking,...",0
1,Augmented Response Synthesis,"Raw outputs from diverse tools (e.g., text, nu...",Outputs received from one or more external too...,The LLM synthesizes information relevant to th...,"A superior, well-informed, and contextually re...","Generating final answers to user queries, summ...",0
2,Tool Learning and Adaptation Strategies,Foundation models struggle to effectively sele...,Integrating foundation models with a dynamic o...,Employ various strategies to enable LLMs to ef...,Enables foundation models to comprehend tool f...,Enabling foundation models to interact with AP...,1
3,Flexible Tool Integration Framework,Integrating Large Language Models (LLMs) with ...,Building LLM-enhanced systems that need to ada...,Design an algorithmic framework that abstracts...,"Achieves high flexibility, generality, and sca...",Any application where LLMs need to be augmente...,1
4,Autonomous Tool Generation,"Manually creating a comprehensive, high-qualit...",When there is a need to rapidly expand an LLM'...,"Enable an LLM to autonomously generate, constr...",Accelerates the development and expansion of t...,"Automatically expanding tool libraries, genera...",1


### Initialize LLM

In [4]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [5]:
summarizing_prompt = """\
You are an expert in **AI Design Pattern Mining**. Your task is to analyze the following cluster of design patterns and determine whether they represent:
1. A single unified pattern (if all share the same underlying principle), or  
2. Multiple related but distinct patterns (if the cluster contains different conceptual directions).

Your goal is to generalize, abstract, or merge the cluster’s content into one or more clear design patterns that accurately capture the shared reasoning or technique.

Give outputs in JSON format:
- patterns: JSON Array
    - Pattern Name: str
    - Problem: str
    - Context: str
    - Solution: str
    - Result: str
    - Uses: str
- thinkings: str
    Explain your thought process and how you arrived at the generalized pattern.

+++

Guidelines:
- If the cluster members express the same core concept, produce **one generalized pattern**.
- If there are different conceptual directions, produce **multiple distinct patterns**.
- Keep pattern names concise but descriptive.
- Avoid redundant or repetitive content across patterns.
- Focus on the underlying mechanism, motivation, or design idea rather than surface wording.
- Use a clear, professional tone suitable for a design patterns repository.

**Your target is to reduce the cluster into the minimal number of patterns that still accurately represent the core ideas.**
Best: one pattern if possible, multiple only if necessary.
Better: fewer patterns, more generalization.
Bad: many patterns with overlapping ideas.

Ensure the outputs contain the patterns field with at least one pattern.

Here is the cluster data:
{cluster_data}
"""

summarizing_prompt_iter = """\
You are an expert in **AI system architecture and design pattern analysis**.
You are given a list of AI design patterns, each containing structured fields such as “Pattern Name,” “Problem,” “Context,” “Solution,” “Result,” and “Uses.”
Your task is to analyze, cluster, and consolidate these patterns to produce a smaller, logically grouped set of merged meta-patterns that preserve all conceptual information but reduce redundancy.
Identify overlap and similarity among patterns — especially those addressing similar problems, contexts, or solution mechanisms
Merge patterns that differ only in scope or granularity, preserving distinctive insights from each under unified headings

Give outputs in JSON format:
- patterns: JSON Array
    - Pattern Name: str
    - Problem: str
    - Context: str
    - Solution: str
    - Result: str
    - Uses: str
- thinkings: str
    Explain your thought process and how you arrived at the generalized pattern.

+++

Guidelines:
- If the cluster members express the same core concept, produce **one generalized pattern**.
- If there are different conceptual directions, produce **multiple distinct patterns**.
- Keep pattern names concise but descriptive.
- Avoid redundant or repetitive content across patterns.
- Focus on the underlying mechanism, motivation, or design idea rather than surface wording.
- Use a clear, professional tone suitable for a design patterns repository.
- If it can not merge the cluster into fewer patterns, you are giving the previous patterns give the same back. Do not refine them.

**Your target is to reduce the cluster into the minimal number of patterns that still accurately represent the core ideas.**
Good: fewer patterns, more generalization.
Bad: many patterns with overlapping ideas.
Bad: refining without reducing the number of patterns.
Bad: changing the meaning of the previous patterns.
Bad: **Generalizing too much that it loses the important details.**
Bad: **Generalizing too less that it misses the common important details.**

Ensure the outputs contain the patterns field with at least one pattern.

Here is the cluster data:
{cluster_data}
"""

summarization_verification_prompt = """\
You are an expert in **AI Design Patterns and JSON validation**. Your task is to verify the correctness of the following summarized design patterns extracted from a cluster of AI design patterns.
The summarized patterns should must be at least one pattern. It should be in valid JSON format. 

Your goal is to ensure that the summarized patterns are:
1. In valid JSON format.
2. Contains all the required fields.
3. At least one pattern is present.

Your Output should be following JSON format:
    "is_valid": bool,
    "issues": str,
    "suggestions": str - In suggestions, provide specific recommendations to fix any identified issues. do not add any bracket or extra characters. just only alphabetical numbered suggestions. keep it short and precise.

++++++++++++++++++++++++++++++++++++
Here is the cluster data:
{cluster_data}

++++++++++++++++++++++++++++++++++++
summarized patterns should contain the following fields (if any field is missing, is_valid should be false):
- patterns: JSON Array
    - Pattern Name: str
    - Problem: str
    - Context: str
    - Solution: str
    - Result: str
    - Uses: str
- thinkings: str
    Explain your thought process and how you arrived at the generalized pattern.

    
If there are no patterns found, set "is_valid" to false and provide appropriate issues and suggestions.
If there are patterns found, check patterns with cluster data for correctness.

Here is the summarized patterns:
{summarized_patterns}
"""

In [9]:
def save_log_summarization(cluster_id, group, summary, verification_result,is_final=False):
    with open('./logs/summarization_log_iter_03.txt', 'a') as f:

        f.write(f"Cluster {cluster_id}\n")
        f.write("="*100 + "\n\n")
        
        # Original Patterns (pretty JSON)
        f.write("Original Patterns:\n")
        f.write(json.dumps(group.to_dict(orient="records"), indent=4, ensure_ascii=False))
        f.write("\n\n")
        
        # Summarized Patterns (pretty JSON)
        f.write("Summarized Patterns:\n")
        f.write(json.dumps(summary, indent=4, ensure_ascii=False))
        f.write("\n\n")
        
        # Verification Result (pretty JSON)
        f.write("Verification Result:\n")
        f.write(json.dumps(verification_result, indent=4, ensure_ascii=False))
        f.write("\n\n")
        if is_final:
            f.write("="*100 + "\n")
        else:
            f.write("-"*100 + "\n")

In [ ]:
def verify_summarization(cluster_df, summarized_patterns):
    # return {
    #     "is_valid": summarized_patterns.get('patterns',[]) and len(summarized_patterns.get('patterns',[]))>0,
    #     "issues": "",
    #     "suggestions": ""
    # }
    cluster_data = cluster_df.to_dict(orient='records')
    prompt = summarization_verification_prompt.format(
        cluster_data=cluster_data,
        summarized_patterns=summarized_patterns
    )
    response = llm.invoke(prompt)
    return parse_json_safe(response.content,delimiter='{}')

def summarize_cluster(cluster_df,retry_count=0,suggestions="",iter=False):
    cluster_data = cluster_df.to_dict(orient='records')
    if iter:
        _prompt = summarizing_prompt_iter
    else:
        _prompt = summarizing_prompt
    if retry_count>0:
        _prompt = _prompt.replace("+++",suggestions)
    prompt = _prompt.format(cluster_data=cluster_data)
    if cluster_df.shape[0]<2:
        print(f" - Only one pattern in cluster {cluster_df['cluster'].iloc[0]}. Skipping summarization.")
        single_pattern = {
            "patterns": [
                {
                    "Pattern Name": cluster_df['Pattern Name'].iloc[0],
                    "Problem": cluster_df['Problem'].iloc[0],
                    "Context": cluster_df['Context'].iloc[0],
                    "Solution": cluster_df['Solution'].iloc[0],
                    "Result": cluster_df['Result'].iloc[0],
                    "Uses": cluster_df['Uses'].iloc[0]
                }
            ],
            "thinkings": "Only one pattern present; no summarization needed."
        }
        save_log_summarization(
            cluster_id=cluster_df['cluster'].iloc[0],
            group=cluster_df,
            summary=single_pattern,
            verification_result={
                "is_valid": True,
                "issues": "",
                "suggestions": ""
            },
            is_final=True
        )
        return single_pattern
    response = llm.invoke(prompt)
    verification = verify_summarization(cluster_df, json.dumps(parse_json_safe(response.content,delimiter='{}'),indent=2))
    summarization = parse_json_safe(response.content,delimiter='{}')
    
    if verification.get('is_valid') or retry_count>=3:
        save_log_summarization(
            cluster_id=cluster_df['cluster'].iloc[0],
            group=cluster_df,
            summary=summarization,
            verification_result=verification,
            is_final=True
        )
        return summarization
    else:
        save_log_summarization(
            cluster_id=cluster_df['cluster'].iloc[0],
            group=cluster_df,
            summary=summarization,
            verification_result=verification,
            is_final=False
        )
        print(f" - Retrying summarization for cluster with {len(cluster_df)} patterns. Retry count: {retry_count+1}")
        print(" - Issues:", verification.get('issues',''))
        print(" - Suggestions:", verification.get('suggestions',''))
        return summarize_cluster(cluster_df,retry_count=retry_count+1,suggestions=verification.get('suggestions',''))


In [8]:
sample_output = """{
    "verification": {
        "is_valid": True,
        "issues": "",
        "suggestions": ""
    }
}"""

verify_summarization(data[data["cluster"] == 0],sample_output)

{'patterns': [{'Pattern Name': 'LLM-Tool Orchestration and Augmentation',
   'Problem': 'Large Language Models (LLMs) are inherently limited by their static training data, lacking real-time knowledge, precise computation, and direct action capabilities. This leads to factual inaccuracies (hallucinations), inability to perform complex numerical or logical reasoning, and an inability to interact with dynamic external environments or perform real-world actions, hindering their ability to solve complex, real-world tasks.',
   'Context': 'AI systems, particularly LLMs, are tasked with complex real-world problems that demand capabilities beyond their internal knowledge or reasoning. These tasks often require high factual accuracy, access to current or specialized data, precise computations, or the ability to interpret varied user inputs and execute actions in external environments (e.g., web services, databases, APIs, specialized software, or physical devices).',
   'Solution': "The LLM acts

### Proccess Clusters

In [13]:
summarizations = []
for cluster_id, group in data.groupby('cluster'):
    print(f"Processing Cluster {cluster_id} with {len(group)} patterns...")
    summary = summarize_cluster(group,iter=True)
    print(f" - {len(summary.get('patterns',[]))} patterns")
    print(" - ", summary.get('patterns',''))
    summarizations.extend(summary.get('patterns',[]))

Processing Cluster 0 with 2 patterns...
 - 1 patterns
 -  [{'Pattern Name': 'LLM-Tool Orchestration and Augmentation', 'Problem': "Large Language Models (LLMs) are inherently limited by their static training data, lacking real-time knowledge, precise computation, and direct action capabilities. This leads to factual inaccuracies (hallucinations), inability to perform complex numerical or logical reasoning, and an inability to interact with dynamic external environments or perform real-world actions, hindering their ability to solve complex, real-world tasks. Furthermore, the raw, diverse outputs from external tools are often complex, varied, and impractical to present directly to users, requiring sophisticated synthesis and integration with the LLM's internal knowledge to form a coherent, user-friendly response.", 'Context': "AI systems, particularly LLMs, are tasked with complex real-world problems that demand capabilities beyond their internal knowledge or reasoning. These tasks ofte

KeyError: 'pattern_name'

In [31]:
summarize_df = pd.DataFrame(summarizations)
os.makedirs("./summarized_patterns/",exist_ok=True)
summarize_df.to_csv('./summarized_patterns/summarized_patterns_iter_02_v0.2.csv', index=False)
json.dump(summarizations, open('./summarized_patterns/summarized_patterns_iter_02_v0.2.json','w'), indent=2, ensure_ascii=False)

In [37]:
summarize_df.shape

(68, 6)

In [38]:
summarize_df.tail()

,Pattern Name,Problem,Context,Solution,Result,Uses
63,Enhanced Input Understanding,Large Language Models (LLMs) often struggle to...,"When the accuracy, relevance, or quality of th...",The LLM is explicitly instructed or designed t...,Significantly enhances the LLM's internal repr...,"Complex question answering, detailed reasoning..."
64,Ambiguity-Robust Demonstrations,Improving an LLM's In-Context Learning (ICL) p...,When the LLM is expected to process inputs tha...,The prompt is engineered to include demonstrat...,Enhances the LLM's ability to robustly handle ...,"Improving ICL performance for ambiguous tasks,..."
65,Multilingual Contextual Augmentation and Adapt...,Large Language Models (LLMs) may produce subop...,When interacting with LLMs in languages other ...,Enhance the LLM's understanding and generation...,"Improved output quality, accuracy, and context...","Machine translation, content localization, cul..."
66,Multi-Stage Refinement for High-Quality Multil...,"Achieving high-quality, nuanced multilingual o...","When the quality, accuracy, stylistic consiste...",Decompose the multilingual generation task int...,"Produces higher-quality, more accurate, fluent...","High-stakes machine translation, creative cont..."
67,Segmented Processing for Large Multilingual In...,Processing very long texts or large datasets w...,"When processing extensive documents, articles,...","Divide the large source input into smaller, ma...",Enables accurate and consistent processing of ...,"Machine translation of long documents, summari..."


In [33]:
import json

def json_to_collapsible_md(input_file, output_file):
    # Read the JSON data
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    md_content = "# 📘 AI Design Patterns\n\n"

    for item in data:
        pattern_name = item.get("Pattern Name", "Unnamed Pattern")
        md_content += f"<details>\n<summary><b>{pattern_name}</b></summary>\n\n"

        for key, value in item.items():
            if key != "Pattern Name":
                md_content += f"### {key}\n\n{value.strip()}\n\n"

        md_content += "</details>\n\n---\n\n"

    # Write Markdown content to file
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(md_content)

    print(f"✅ Markdown file created successfully: {output_file}")


In [35]:
json_to_collapsible_md("./summarized_patterns/summarized_patterns_iter_02_v0.2.json", "../patterns/RAG/patterns_v1.md")

✅ Markdown file created successfully: ../patterns/RAG/patterns_v1.md
